In [1]:
# -*- coding: utf-8 -*-
"""
════════════════════════════════════════════════════════════════════════════
П.7 «Итогов второй встречи с оппонентом»
Выбор лучшего отведения по критерию чистоты определения зубцов P, Q, R, S, T

ЕДИНЫЙ ФАЙЛ. Ничего импортировать из соседних файлов не нужно.

Установка:
    pip install numpy scipy pyedflib neurokit2 matplotlib

Запуск из командной строки:
    python П7_выбор_отведения_единый_код.py /путь/к/записям [--out results_p7]

Запуск в Jupyter / Google Colab (выполнить ячейку целиком, затем):
    res = run_p7('/путь/к/записям', out_dir='results_p7')

В папке с записями должны лежать файлы 00*_ecg.edf (три отведения).
Разметка каждой записи сохраняется в out_dir/marks/, поэтому прерванный
расчёт продолжается с места остановки (полный расчёт занимает несколько минут,
до 10 — в зависимости от компьютера; основное время — независимый разметчик NeuroKit2).

Что делает расчёт
  1. В каждом отведении находит R, P и T (детекторы, общие для всех пунктов),
     Q и S — исправленным детектором по ЭКГ без дрейфа изолинии.
  2. Независимо размечает P и T методом NeuroKit2 (вейвлет-разметка).
  3. Для каждого зубца считает долю циклов, где он определён ЧИСТО:
       P — найден, стоит на своём месте и совпадает с NeuroKit2 (до 20 мс);
       T — найден и совпадает с NeuroKit2 (до 30 мс);
       Q, S — найдены и стоят на своём месте (до 12 мс);
       R — положительный R есть и опорная точка не перескакивает (до 10 мс).
     «На своём месте» — расстояние до опорной точки комплекса отличается от
     скользящей медианы по 21 циклу не больше допуска.
  4. Балл отведения = среднее пяти долей × полнота обнаружения комплексов.
  5. Сравнивает два лучших отведения: 95 % интервал разницы баллов
     (бутстреп блоками по 20 циклов). Если интервал включает ноль, отведения
     равноценны, и выбирается отведение с большим отношением сигнал/шум.
  6. Проверяет детекторы и критерий на синтетических ЭКГ с известной
     разметкой, сравнивает с исходным критерием и проверяет устойчивость
     выбора к настройкам.
════════════════════════════════════════════════════════════════════════════
"""
!pip install numpy scipy pyedflib neurokit2 matplotlib
import argparse
import csv
import glob
import itertools
import json
import math
import os
import pickle
import warnings

import numpy as np
import pyedflib
from scipy.signal import butter, filtfilt, find_peaks
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

CHANNELS = [0, 1, 2]
NAMES = {0: 'I', 1: 'II', 2: 'III'}   # стандартные отведения (каналы записи 0, 1, 2)


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 1. Детекторы R, P, T (общие для всех пунктов)
# ═══════════════════════════════════════════════════════════════════════




def detect_r(sig, fs, band=(5.0, 20.0), integ_ms=120,
             min_rr_ms=300, win_s=10.0, k=0.35, refine_ms=100):
    x = np.asarray(sig, float)
    x = x - np.median(x)

    # 1) полосовой фильтр — полоса энергии QRS
    ny = fs/2
    b, a = butter(3, [band[0]/ny, min(band[1]/ny, 0.99)], btype='band')
    xf = filtfilt(b, a, x)

    # 2) производная + возведение в квадрат (подчёркивает крутые фронты QRS)
    d = np.diff(xf, append=xf[-1])
    sq = d**2

    # 3) интегрирование скользящим окном ~ длительность QRS
    w = max(3, int(round(integ_ms/1000*fs)))
    integ = np.convolve(sq, np.ones(w)/w, mode='same')

    # 4) ЛОКАЛЬНО адаптивный порог: доля от локального максимума
    #    (окно win_s секунд, шаг — половина окна)
    n = len(integ)
    W = max(int(win_s*fs), w*4)
    thr = np.zeros(n)
    for start in range(0, n, W//2):
        end = min(start+W, n)
        seg = integ[start:end]
        if len(seg) == 0:
            continue
        local = k*np.percentile(seg, 98)      # 98-й процентиль ≈ уровень QRS
        thr[start:end] = np.maximum(thr[start:end], local)
    thr[thr == 0] = k*np.percentile(integ, 98)

    # 5) поиск пиков с рефрактерным периодом
    dist = max(1, int(round(min_rr_ms/1000*fs)))
    cand, _ = find_peaks(integ, height=thr, distance=dist)

    # 6) уточнение положения по максимуму |полосового сигнала|
    half = max(2, int(round(refine_ms/1000*fs/2)))
    out = []
    for c in cand:
        a0, b0 = max(0, c-half), min(len(xf), c+half+1)
        out.append(a0 + int(np.argmax(np.abs(xf[a0:b0]))))
    return np.array(sorted(set(out)), dtype=int), xf


def suppress_qrs(sig, r_idx, fs, before_ms=50, after_ms=100):
    out = sig.copy()
    b = int(round(before_ms/1000*fs)); a = int(round(after_ms/1000*fs))
    for r in r_idx:
        i0, i1 = max(0, r-b), min(len(sig)-1, r+a)
        if i1-i0 < 2: continue
        out[i0:i1+1] = np.interp(np.arange(i0, i1+1), [i0, i1], [sig[i0], sig[i1]])
    return out


def halfwave(lam):
    """Вейвлет «полуволна»: положительный полупериод синуса
       с симметричными отрицательными крыльями (нулевое среднее)."""
    if lam % 2 == 0: lam += 1
    pos = np.sin(np.linspace(0, math.pi, lam))
    area = float(np.trapezoid(pos)) if hasattr(np, 'trapezoid') else float(np.trapz(pos))
    wing = max(1, lam//4)
    w = np.zeros(lam + 2*wing)
    w[:wing] = -area/(2*wing)
    w[wing:wing+lam] = pos
    w[wing+lam:] = -area/(2*wing)
    return w - w.mean()


def detect_pt(sig, r_idx, fs, wave_ms=85, thr_frac=0.3, local_s=30.0, noise_k=10.0,
              tol_ms=20.0, win=21):
    """Возвращает индексы P и T (по одному на кардиоцикл, если найдены).
    Отклик — свёртка ЭКГ (с подавленными комплексами QRS) с вейвлетом «полуволна».
    T и основной поиск P: порог = thr_frac · 99-й процентиль положительного
    отклика по всей записи (без участков с аномально длинными RR).
    Дополнительный поиск P (для записей, где P в одной из фаз заметно меньше):
    порог по откликам в окнах поиска P в скользящем окне ±local_s секунд, но не
    ниже noise_k · уровня шума; такая находка принимается, только если расстояние
    P–R отличается от скользящей медианы по win циклам не больше tol_ms."""
    r_idx = np.asarray(r_idx, int)
    ecg = suppress_qrs(sig, r_idx, fs)
    lam = max(5, int(round(wave_ms/1000*fs)))
    w = halfwave(lam)
    conv = np.convolve(ecg, w[::-1], mode='same')
    n = len(conv)

    # порог по «здоровым» участкам (исключаем аномально длинные RR)
    rr = np.diff(r_idx)
    med = np.median(rr) if len(rr) else 0
    ok = np.ones(n, bool)
    for k in range(len(rr)):
        if med and rr[k] > 1.8*med:
            ok[r_idx[k]:r_idx[k+1]] = False
    pos = conv[(conv > 0) & ok]
    thr = thr_frac*np.percentile(pos, 99) if len(pos) else 0

    # окна поиска P и T
    wins = []
    for k in range(len(r_idx)):
        r = r_idx[k]
        r_prev = r_idx[k-1] if k > 0 else max(0, r-int(fs))
        r_next = r_idx[k+1] if k < len(r_idx)-1 else min(len(sig)-1, r+int(fs))
        # P: непосредственно перед QRS (P предшествует R на 100–300 мс)
        rr_p = r - r_prev
        lo = max(r_prev + rr_p//2, r - int(0.35*fs))
        ps, pe = lo, r - max(1, int(0.06*fs))
        # T: после QRS, до начала следующего цикла
        rr_n = r_next - r
        ts = r + max(int(0.12*fs), rr_n//8)
        te = min(r + int(0.6*rr_n), r_next - int(0.08*fs))
        wins.append((ps, pe, ts, te))

    P, T = [], []
    for ps, pe, ts, te in wins:
        if ps < pe <= len(conv):
            seg = conv[ps:pe]
            P.append(ps + int(np.argmax(seg)) if len(seg) and seg.max() > thr else -1)
        else:
            P.append(-1)
        if ts < te <= len(conv):
            seg = conv[ts:te]
            T.append(ts + int(np.argmax(seg)) if len(seg) and seg.max() > thr else -1)
        else:
            T.append(-1)
    P = np.array(P); T = np.array(T)

    # дополнительный поиск P: порог по окнам P в скользящем окне со страховкой от шума
    pmask = np.zeros(n, bool)
    for ps, pe, _, _ in wins:
        if ps < pe <= n:
            pmask[ps:pe] = True
    d = np.diff(ecg)
    noise = 1.4826*np.median(np.abs(d - np.median(d)))/np.sqrt(2)*np.sqrt(np.sum(w**2))
    L = int(local_s*fs)
    P2 = np.full(len(r_idx), -1)
    for k, (ps, pe, _, _) in enumerate(wins):
        if not (ps < pe <= n):
            continue
        i = r_idx[k]; a, b = max(0, i-L), min(n, i+L)
        sg = conv[a:b]; mk = (sg > 0) & ok[a:b] & pmask[a:b]
        thr_p = thr_frac*np.percentile(sg[mk], 99) if mk.sum() > 10 else thr
        seg = conv[ps:pe]
        if len(seg) and seg.max() > max(thr_p, noise_k*noise):
            P2[k] = ps + int(np.argmax(seg))
    off = np.where(P2 >= 0, (r_idx - P2)/fs*1000, np.nan)
    med_off = np.full(len(off), np.nan); h = win//2
    for i in range(len(off)):
        s = off[max(0, i-h):i+h+1]; s = s[np.isfinite(s)]
        if len(s) >= 3:
            med_off[i] = np.median(s)
    add = (P < 0) & (P2 >= 0) & (np.abs(off - med_off) <= tol_ms)
    P[add] = P2[add]
    return P, T, ecg, conv


def read_ecg(path, ch=0):
    f = pyedflib.EdfReader(path)
    fs = float(f.getSampleFrequency(ch))
    sig = f.readSignal(ch).astype(float)
    f.close()
    return sig, fs


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 2. Исправленный детектор Q и S
# ═══════════════════════════════════════════════════════════════════════

def ecg_band(sig, fs, lo=0.5, hi=40.0):
    """ЭКГ без дрейфа изолинии и высокочастотного шума (полоса 0,5–40 Гц)."""
    b, a = butter(2, [lo/(fs/2), hi/(fs/2)], 'band')
    return filtfilt(b, a, np.asarray(sig, float))


def tp_noise(y, r_idx, fs):
    """Уровень шума: стандартное отклонение сигнала на изоэлектрическом участке TP
    (от 55 % интервала RR после R до 25 % RR перед следующим R) после вычитания
    линейного тренда; медиана по циклам. Работает и для гладкого сигнала."""
    vals = []
    r_idx = np.asarray(r_idx, int)
    for a, b in zip(r_idx[:-1], r_idx[1:]):
        rr = b - a; s0, s1 = a + int(0.55*rr), b - int(0.25*rr)
        if s1 - s0 < int(0.06*fs): continue
        seg = y[s0:s1]; t = np.arange(len(seg))
        vals.append(np.std(seg - np.polyval(np.polyfit(t, seg, 1), t)))
    return float(np.median(vals)) if vals else 0.0


def detect_qrs_points(y, r_idx, fs, search_ms=40, q_ms=50, s_ms=70, k=4.0, min_amp=25.0):
    """R, Q, S по ЭКГ после удаления дрейфа изолинии (y).
    R — положительный максимум в окне ±search_ms вокруг найденного комплекса; если
    опорная точка лежит на отрицательном отклонении (дно S), R ищется только до неё.
    Q / S — минимум в окне перед / после R. Зубец засчитывается, если минимум лежит
    строго внутри окна (на краю окна — это склон, а не зубец) и ниже изолинии
    больше чем на порог. Изолиния — медиана сигнала за предыдущий сердечный цикл;
    порог — max(k * шум, min_amp), шум — по участкам TP (tp_noise)."""
    y = np.asarray(y, float); n = len(y); r_idx = np.asarray(r_idx, int)
    w = lambda ms: max(2, int(round(ms/1000*fs)))
    thr = max(k*tp_noise(y, r_idx, fs), min_amp)
    R, Q, S = [], [], []
    for i, r in enumerate(r_idx):
        c0 = r_idx[i-1] if i > 0 else max(0, r - w(800))
        base = np.median(y[c0:r]) if r - c0 > 10 else np.median(y[max(0, r-w(800)):r+1])
        if y[r] < base:
            # опорная точка на отрицательном отклонении (дно S в комплексе rS / QS):
            # R по определению предшествует S, поэтому ищем его только до опорной точки
            a0, a1 = max(0, r-w(search_ms+20)), r
        else:
            a0, a1 = max(0, r-w(search_ms)), min(n, r+w(search_ms)+1)
        if a1 - a0 < 2:
            R.append(-1); Q.append(-1); S.append(-1); continue
        rp = a0 + int(np.argmax(y[a0:a1]))
        R.append(rp if y[rp]-base > thr else -1)
        def trough(s0, s1):
            seg = y[s0:s1]
            if len(seg) < 3: return -1
            j = int(np.argmin(seg))
            if j == 0 or j == len(seg)-1: return -1
            return s0+j if base-seg[j] > thr else -1
        Q.append(trough(max(0, rp-w(q_ms)), rp))
        S.append(trough(rp+1, min(n, rp+w(s_ms)+1)))
    return np.array(R), np.array(Q), np.array(S)



# Исходный детектор Q и S (первый опробованный вариант) — только для сравнения.
# Ищет зубцы по сигналу после полосового фильтра 5–20 Гц, где «звон» фильтра
# даёт отрицательные провалы около любого острого пика.
def detect_qs_original(sig, r_idx, fs, q_ms=40, s_ms=60, k=2.5, baseline_ms=40):
    """Q — локальный минимум в окне q_ms перед R, S — в окне s_ms после R.
    "Найден", если провал относительно ближайшей изолинии (участок TP,
    оценённый как медиана сигнала в окне baseline_ms до Q-окна) превышает
    k * (локальный шум, MAD за тот же участок изолинии)."""
    r_idx = np.asarray(r_idx, int)
    qw = max(2, int(round(q_ms/1000*fs)))
    sw = max(2, int(round(s_ms/1000*fs)))
    bw = max(2, int(round(baseline_ms/1000*fs)))
    Q, S = [], []
    for r in r_idx:
        # --- Q ---
        b0, b1 = max(0, r-qw-bw), max(0, r-qw)
        q0, q1 = max(0, r-qw), r
        if b1 > b0 and q1 > q0:
            base = np.median(sig[b0:b1])
            noise = np.median(np.abs(sig[b0:b1]-base)) * 1.4826 + 1e-9
            seg = sig[q0:q1]
            j = int(np.argmin(seg))
            depth = base - seg[j]
            Q.append(q0+j if depth > k*noise else -1)
        else:
            Q.append(-1)
        # --- S ---
        s0, s1 = r, min(len(sig), r+sw)
        b0s, b1s = min(len(sig), r+sw), min(len(sig), r+sw+bw)
        if s1 > s0 and b1s > b0s:
            base = np.median(sig[b0s:b1s])
            noise = np.median(np.abs(sig[b0s:b1s]-base)) * 1.4826 + 1e-9
            seg = sig[s0:s1]
            j = int(np.argmin(seg))
            depth = base - seg[j]
            S.append(s0+j if depth > k*noise else -1)
        else:
            S.append(-1)
    return np.array(Q), np.array(S)



# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 3. Разметка отведения двумя методами
# ═══════════════════════════════════════════════════════════════════════

def nk_delineate(sig, r, fs):
    """Независимая разметка P и T методом NeuroKit2 (вейвлет-разметка).
    Отсутствующая точка = -1. Возвращает None, если разметка не удалась."""
    import neurokit2 as nk
    try:
        clean = nk.ecg_clean(sig, sampling_rate=int(fs))
        _, w = nk.ecg_delineate(clean, r, sampling_rate=int(fs), method='dwt')
        f = lambda k: np.nan_to_num(np.array(w[k], float), nan=-1).astype(int)
        return dict(P=f('ECG_P_Peaks'), T=f('ECG_T_Peaks'))
    except Exception:
        return None


def mark_lead(sig, fs, use_nk=True):
    """Полная разметка одного отведения."""
    r, xf = detect_r(sig, fs)
    P, T, _, _ = detect_pt(sig, r, fs)
    y = ecg_band(sig, fs)
    R, Q, S = detect_qrs_points(y, r, fs)
    Qf, Sf = detect_qs_original(xf, r, fs)
    w = int(0.06*fs)
    amp = float(np.median([np.ptp(y[max(0, i-w):i+w]) for i in r])) if len(r) else 0.0
    noise = tp_noise(y, r, fs)
    return dict(fs=fs, r=r, P=P, T=T, R=R, Q=Q, S=S, Q_filt=Qf, S_filt=Sf,
                nk=nk_delineate(sig, r, fs) if use_nk else None,
                snr=amp/noise if noise > 0 else float('inf'))


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 4. Критерий чистоты и выбор отведения
# ═══════════════════════════════════════════════════════════════════════
TOL_MS={'P':20,'Q':12,'R':10,'S':12,'T':30}
def rolling_nanmedian(x,win=21):
    out=np.full(len(x),np.nan); h=win//2
    for i in range(len(x)):
        s=x[max(0,i-h):i+h+1]; s=s[np.isfinite(s)]
        if len(s)>=3: out[i]=np.median(s)
    return out
def consistent(idx,ref,fs,tol_ms,win=21):
    idx=np.asarray(idx,int); ref=np.asarray(ref,int); L=min(len(idx),len(ref)); idx,ref=idx[:L],ref[:L]
    found=(idx>=0)&(ref>=0); off=np.where(found,(idx-ref)/fs*1000,np.nan)
    return found&(np.abs(off-rolling_nanmedian(off,win))<=tol_ms)
def agree(a,b,fs,tol_ms):
    a=np.asarray(a,int); b=np.asarray(b,int); L=min(len(a),len(b))
    return (a[:L]>=0)&(b[:L]>=0)&(np.abs(a[:L]-b[:L])<=tol_ms/1000*fs)
def purity(d,tol_scale=1.0,win=21,use_nk=True):
    """Доля циклов, где каждый зубец определён чисто. d — разметка одного отведения.
    P и T дополнительно подтверждаются независимым разметчиком NeuroKit2 (для них
    его точки совпадают с нашими по определению); для Q и S он не используется:
    его точка Q лежит на 13 мс раньше дна зубца, а точка S в части отведений
    отсутствует или смещена на сегмент ST."""
    fs=d['fs']; r=np.asarray(d['r'],int); n=len(r); t=lambda w: TOL_MS[w]*tol_scale
    cut=lambda x: np.asarray(x)[:n]
    pur={}
    pur['R']=consistent(d['R'],r,fs,t('R'),win)                       # положит. R есть и опорная точка не перескакивает
    pur['Q']=consistent(d['Q'],r,fs,t('Q'),win)
    S=consistent(d['S'],r,fs,t('S'),win); P=consistent(d['P'],r,fs,t('P'),win); T=(cut(d['T'])>=0)
    if use_nk and d.get('nk') is not None:
        nk=d['nk']; P=P&agree(d['P'],nk['P'],fs,t('P'))[:n]; T=T&agree(d['T'],nk['T'],fs,t('T'))[:n]
    pur['P'],pur['S'],pur['T']=P,S,T
    return {w:np.asarray(pur[w])[:n] for w in 'PQRST'}
def lead_score(d,max_n,how='mean',**kw):
    pur=purity(d,**kw); rates={w:float(pur[w].mean()) for w in 'PQRST'}; rate_r=len(d['r'])/max_n
    v=np.array([rates[w] for w in 'PQRST'])
    allfive=float(np.all([pur[w] for w in 'PQRST'],axis=0).mean())
    sc={'mean':v.mean(),'all':allfive,'geo':float(np.exp(np.mean(np.log(np.maximum(v,0.01)))))}[how]*rate_r
    return sc,rates,allfive,rate_r


def original_score(d, max_n):
    """Исходный критерий (первый опробованный вариант): доля циклов, где найдены P, Q, S, T
    (Q и S — по отфильтрованному сигналу) × полнота обнаружения R."""
    ok = (d['P'] >= 0) & (d['Q_filt'] >= 0) & (d['S_filt'] >= 0) & (d['T'] >= 0)
    return float(ok.mean() * len(d['r']) / max_n)


def per_beat_score(d, max_n):
    p = purity(d)
    return np.mean([p[w] for w in 'PQRST'], axis=0) * len(d['r']) / max_n


def diff_ci(d1, d2, max_n, B=1000, block=20, seed=0):
    """95 % интервал разницы баллов двух отведений (бутстреп блоками по 20 циклов)."""
    a, b = per_beat_score(d1, max_n), per_beat_score(d2, max_n)
    L = min(len(a), len(b)); diff = a[:L] - b[:L]
    rng = np.random.default_rng(seed); nb = L // block
    bs = [diff[np.concatenate([np.arange(k*block, (k+1)*block) for k in rng.integers(0, nb, nb)])].mean()
          for _ in range(B)]
    return tuple(float(v) for v in np.percentile(bs, [2.5, 97.5]))


def choose_lead(rec, how='mean', **kw):
    """rec: {канал: разметка}. Возвращает выбор и подробности."""
    mx = max(len(rec[c]['r']) for c in rec)
    sc = {c: lead_score(rec[c], mx, how, **kw)[0] for c in rec}
    order = sorted(sc, key=sc.get, reverse=True)
    lo, hi = diff_ci(rec[order[0]], rec[order[1]], mx)
    tie = lo <= 0
    pick = max(order[:2], key=lambda c: rec[c]['snr']) if tie else order[0]
    return dict(pick=pick, scores=sc, leader=order[0], second=order[1], ci=(lo, hi), tie=bool(tie))


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 5. Синтетические ЭКГ с известной разметкой
# ═══════════════════════════════════════════════════════════════════════

FS_SYN = 500.0
BASE = dict(p=120, q=150, s=300, t=300, r=1200, noise=12, wander=60, scale=1.0, mod=0.0)
DEFECTS = {'нет Q': dict(q=0), 'нет S': dict(s=0), 'перевёрнутый T': dict(t=-300),
           'слабый P и шум': dict(p=35, noise=35), 'нет P': dict(p=0),
           'перескок R/S': dict(r=1000, s=1000, mod=0.3),
           'низкая амплитуда и шум': dict(scale=0.2, noise=25)}
TRUTH_TOL_MS = {'P': 20, 'Q': 12, 'R': 10, 'S': 12, 'T': 30}


def synth_ecg(cfg, seed, n=80, fs=FS_SYN):
    """Синтетическая ЭКГ из гауссовых зубцов. Возвращает сигнал и истинные
    положения зубцов (в отсчётах). mod — дыхательная модуляция R и S в
    противофазе (воспроизводит перескок опорной точки между R и S)."""
    rng = np.random.default_rng(seed)
    t = np.arange(0, n*0.85 + 1.5, 1/fs); x = np.zeros_like(t)
    g = lambda c, a, w: a*np.exp(-0.5*((t-c)/w)**2)
    truth = {k: [] for k in 'PQRST'}; c = 0.6
    for k in range(n):
        mo = cfg['mod']*np.sin(2*np.pi*k/4.7)
        waves = [('P', c-0.16, cfg['p'], 0.022), ('Q', c-0.022, -cfg['q'], 0.007),
                 ('R', c, cfg['r']*(1+mo), 0.010), ('S', c+0.028, -cfg['s']*(1-mo), 0.009),
                 ('T', c+0.30, cfg['t'], 0.045)]
        for name, cc, a, w in waves:
            if a:
                x += g(cc, a, w); truth[name].append(cc*fs)
        c += 0.85 + 0.04*np.sin(2*np.pi*k/12) + 0.01*rng.standard_normal()
    x = cfg['scale']*x + cfg['noise']*rng.standard_normal(len(t)) \
        + cfg['wander']*np.sin(2*np.pi*0.25*t + rng.uniform(0, 6))
    return x, {k: np.array(v) for k, v in truth.items()}


def detector_check():
    """Проверка детектора на синтетике: доля найденных зубцов (если зубец есть)
    или доля ложных срабатываний (если зубца нет); для Q и S — оба детектора."""
    cases = [('все зубцы', {}), ('все зубцы, шум и дрейф', dict(noise=40, wander=150)),
             ('нет Q', dict(q=0)), ('нет S', dict(s=0)), ('нет Q и S', dict(q=0, s=0)),
             ('нет P', dict(p=0)), ('комплекс rS', dict(q=0, r=250, s=900)),
             ('сильный шум', dict(noise=80, wander=150))]
    rows = []
    for title, extra in cases:
        x, tr = synth_ecg(dict(BASE, **extra), seed=0)
        d = mark_lead(x, FS_SYN, use_nk=False)
        row = {'случай': title}
        for w, key in [('P', 'P'), ('Q', 'Q'), ('S', 'S'), ('T', 'T'), ('Q_исх', 'Q_filt'), ('S_исх', 'S_filt')]:
            found = np.asarray(d[key]); found = found[found >= 0].astype(float)
            true = tr[w[0]]
            if len(true):
                hit = sum(np.min(np.abs(found - t0)) <= TRUTH_TOL_MS[w[0]]/1000*FS_SYN for t0 in true) if len(found) else 0
                row[w] = ('найдено', hit/len(true))
            else:
                row[w] = ('ложных', len(found)/len(d['r']))
        rows.append(row)
    return rows


def selection_check(seeds=(0, 1)):
    """42 синтетические трёхканальные записи: одно чистое отведение и два с
    дефектами (все пары из семи дефектов, две реализации шума). Считается,
    как часто критерий выбирает чистое отведение; ничья с дефектным
    отведением считается неудачей."""
    out = {'новый': [0, 0, {}], 'исходный': [0, 0, {}], 'новый без NeuroKit2': [0, 0, {}]}
    for seed in seeds:
        for a, b in itertools.combinations(DEFECTS, 2):
            cfgs = [dict(BASE), dict(BASE, **DEFECTS[a]), dict(BASE, **DEFECTS[b])]
            leads = [mark_lead(synth_ecg(cfg, seed*10+i)[0], FS_SYN) for i, cfg in enumerate(cfgs)]
            mx = max(len(d['r']) for d in leads)
            scorers = {'новый': lambda d: lead_score(d, mx, 'mean')[0],
                       'исходный': lambda d: original_score(d, mx),
                       'новый без NeuroKit2': lambda d: lead_score(d, mx, 'mean', use_nk=False)[0]}
            for name, fn in scorers.items():
                sc = np.array([fn(d) for d in leads]); top = np.where(np.abs(sc - sc.max()) < 1e-9)[0]
                if len(top) > 1 and 0 in top:
                    out[name][1] += 1
                    for i in top:
                        if i:
                            k = [a, b][i-1]; out[name][2][k] = out[name][2].get(k, 0) + 1
                elif top[0] == 0:
                    out[name][0] += 1
    n = 2*len(list(itertools.combinations(DEFECTS, 2)))
    return {k: dict(correct=v[0], ties=v[1], tied_with=v[2], total=n) for k, v in out.items()}


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 6. Расчёт по записям
# ═══════════════════════════════════════════════════════════════════════

def mark_records(data_dir, out_dir):
    """Размечает все записи; каждая запись сохраняется отдельно, поэтому
    повторный запуск продолжает с места остановки."""
    mdir = os.path.join(out_dir, 'marks'); os.makedirs(mdir, exist_ok=True)
    paths = sorted(glob.glob(os.path.join(data_dir, '00*_ecg.edf')))
    if not paths:
        raise SystemExit(f'В папке {data_dir} нет файлов 00*_ecg.edf')
    marks = {}
    for p in paths:
        name = os.path.basename(p)[:4]; f = os.path.join(mdir, name + '.pkl')
        if os.path.exists(f):
            marks[name] = pickle.load(open(f, 'rb')); continue
        rec = {}
        for ch in CHANNELS:
            sig, fs = read_ecg(p, ch=ch); rec[ch] = mark_lead(sig, fs)
        pickle.dump(rec, open(f, 'wb')); marks[name] = rec
        print(f'  размечена запись {name}', flush=True)
    return marks


SENSITIVITY = {'все пять зубцов в одном цикле': dict(how='all'),
               'геометрическое среднее': dict(how='geo'),
               'допуски ×0,5': dict(tol_scale=0.5), 'допуски ×1,5': dict(tol_scale=1.5),
               'окно 11 циклов': dict(win=11), 'окно 41 цикл': dict(win=41)}


def argmax_choice(rec, how='mean', **kw):
    mx = max(len(rec[c]['r']) for c in rec)
    sc = {c: lead_score(rec[c], mx, how, **kw)[0] for c in rec}
    return max(sc, key=sc.get)


def analyse_records(marks):
    rows = []
    for name, rec in marks.items():
        mx = max(len(rec[c]['r']) for c in rec)
        res = choose_lead(rec)
        orig = {c: original_score(rec[c], mx) for c in rec}
        rates = {c: lead_score(rec[c], mx, 'mean')[1] for c in rec}
        rows.append(dict(record=name, pick=res['pick'], orig_pick=max(orig, key=orig.get),
                         scores=res['scores'], orig_scores=orig, rates=rates,
                         leader=res['leader'], second=res['second'], ci=res['ci'], tie=res['tie'],
                         snr={c: rec[c]['snr'] for c in rec}))
    main_arg = {n: argmax_choice(marks[n]) for n in marks}
    sens = {}
    for title, kw in SENSITIVITY.items():
        same = sum(argmax_choice(marks[n], **kw) == main_arg[n] for n in marks)
        sens[title] = same
    return rows, sens


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 7. Рисунки
# ═══════════════════════════════════════════════════════════════════════

def _comma_axes(fig):
    """Десятичная запятая на линейных осях (русская запись чисел)."""
    from matplotlib.ticker import FuncFormatter
    fmt = FuncFormatter(lambda v, _: f'{v:g}'.replace('.', ','))
    for a in fig.axes:
        a.xaxis.set_major_formatter(fmt); a.yaxis.set_major_formatter(fmt)


def fig_filter_artifact(path):
    """Почему исходный детектор Q/S не работал: синтетический комплекс без Q и S
    после фильтра 5–20 Гц и без него."""
    x, _ = synth_ecg(dict(BASE, q=0, s=0, noise=3, wander=0), seed=0, n=6)
    r, xf = detect_r(x, FS_SYN); y = ecg_band(x, FS_SYN)
    Qf, Sf = detect_qs_original(xf, r, FS_SYN); R, Q, S = detect_qrs_points(y, r, FS_SYN)
    i0, i1 = int(r[1]-0.25*FS_SYN), int(r[1]+0.45*FS_SYN); t = np.arange(i0, i1)/FS_SYN
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    for a, sig, q, s, title in [(ax[0], xf, Qf, Sf, 'а — после фильтра 5–20 Гц (исходный детектор)'),
                                (ax[1], y, Q, S, 'б — ЭКГ без дрейфа (исправленный детектор)')]:
        a.plot(t, sig[i0:i1], 'k', lw=1)
        qi, si = q[1], s[1]
        if qi >= 0: a.plot(qi/FS_SYN, sig[qi], 'o', color='#1565c0', ms=8, label='отмечен Q')
        if si >= 0: a.plot(si/FS_SYN, sig[si], 'o', color='#2e7d32', ms=8, label='отмечен S')
        a.set_title(title, fontsize=10); a.set_xlabel('время, с'); a.grid(alpha=0.3)
        if qi >= 0 or si >= 0: a.legend(fontsize=8)
        else: a.text(0.5, 0.9, 'Q и S не отмечены', transform=a.transAxes, ha='center', fontsize=9)
    ax[0].set_ylabel('мкВ')
    _comma_axes(fig); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def fig_record(data_dir, rec, name, t0, dur, path, picked=None, orig=None):
    """Три отведения записи на ЭКГ без дрейфа с метками зубцов."""
    fig, ax = plt.subplots(3, 1, figsize=(12, 7.5), sharex=True)
    colors = {'R': '#c62828', 'Q': '#1565c0', 'S': '#2e7d32', 'P': '#ef6c00', 'T': '#6a1b9a'}
    for k, ch in enumerate(CHANNELS):
        sig, fs = read_ecg(os.path.join(data_dir, f'{name}_ecg.edf'), ch=ch); y = ecg_band(sig, fs)
        d = rec[ch]; i0, i1 = int(t0*fs), int((t0+dur)*fs)
        ax[k].plot(np.arange(i0, i1)/fs, y[i0:i1], 'k', lw=0.8)
        for key, col in colors.items():
            idx = np.asarray(d[key]); idx = idx[(idx >= i0) & (idx < i1)]
            ax[k].plot(idx/fs, y[idx], 'o', color=col, ms=5, label=key)
        mark = []
        if ch == picked: mark.append('выбрано пайплайном')
        if ch == orig: mark.append('выбрано исходным критерием')
        ax[k].set_title(f'{name}, {NAMES[ch]}' + (' — ' + '; '.join(mark) if mark else ''), fontsize=10)
        ax[k].set_ylabel('мкВ'); ax[k].grid(alpha=0.3)
    ax[0].legend(fontsize=8, ncol=5, loc='upper right'); ax[-1].set_xlabel('время, с')
    _comma_axes(fig); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


# ═══════════════════════════════════════════════════════════════════════
# ЧАСТЬ 8. Общий запуск
# ═══════════════════════════════════════════════════════════════════════

def check_neurokit():
    """NeuroKit2 нужен для независимой проверки зубцов P и T. Если пакета нет,
    расчёт останавливается с понятным сообщением, а не с технической ошибкой."""
    import importlib.util
    if importlib.util.find_spec('neurokit2') is None:
        raise SystemExit(
            '\nНе установлен пакет neurokit2 (бесплатный, нужен для независимой проверки зубцов P и T).\n'
            'Установите его одной командой и запустите расчёт снова:\n'
            '    в командной строке:     pip install neurokit2\n'
            '    в Jupyter / Colab:      !pip install neurokit2\n')


def run_p7(data_dir, out_dir='results_p7', with_synthetic=True):
    check_neurokit()
    os.makedirs(out_dir, exist_ok=True)
    summary = {}
    if with_synthetic:
        print('Проверка детектора на синтетике...')
        summary['detector_check'] = detector_check()
        for row in summary['detector_check']:
            print('  ' + row['случай'].ljust(24) + ' '.join(
                f"{w}: {'наход.' if v[0] == 'найдено' else 'ложн.'} {v[1]*100:.0f}%" for w, v in row.items() if w != 'случай'))
        print('Проверка выбора на 42 синтетических записях...')
        summary['selection_check'] = selection_check()
        for k, v in summary['selection_check'].items():
            print(f"  {k:22s} верно {v['correct']} из {v['total']}, ничьих {v['ties']} {v['tied_with']}")
    print('Разметка записей...')
    marks = mark_records(data_dir, out_dir)
    rows, sens = analyse_records(marks)
    summary['records'] = rows; summary['sensitivity'] = sens
    print('\nзапись | было | стало | баллы I/II/III | 95 % интервал разницы двух лучших')
    for r in rows:
        s = r['scores']
        print(f"{r['record']} {NAMES[r['orig_pick']]:5s} {NAMES[r['pick']]:5s} " +
              ' '.join(f'{s[c]*100:5.1f}' for c in CHANNELS) +
              f" | {NAMES[r['leader']]}–{NAMES[r['second']]} [{r['ci'][0]*100:.1f}; {r['ci'][1]*100:.1f}]" +
              (' равноценны → по сигнал/шум' if r['tie'] else '') +
              ('' if r['pick'] == r['orig_pick'] else '  ← изменён'))
    changed = [r['record'] for r in rows if r['pick'] != r['orig_pick']]
    counts = {NAMES[c]: sum(r['pick'] == c for r in rows) for c in CHANNELS}
    summary['changed'] = changed; summary['counts'] = counts
    print(f'\nИзменено: {len(changed)} из {len(rows)}: {", ".join(changed)}')
    print('Выбрано: ' + ', '.join(f'{k} — {v}' for k, v in counts.items()))
    print('Устойчивость (совпадение с основным вариантом):')
    for k, v in sens.items():
        print(f'  {k:32s} {v} из {len(rows)}')

    fig_filter_artifact(os.path.join(out_dir, 'p7_fig1_filter_artifact.png'))
    byname = {r['record']: r for r in rows}
    for k, (name, t0) in enumerate([('0025', 100.0), ('0023', 150.0)], start=2):
        if name in marks:
            fig_record(data_dir, marks[name], name, t0, 2.6, os.path.join(out_dir, f'p7_fig{k}_{name}.png'),
                       picked=byname[name]['pick'], orig=byname[name]['orig_pick'])

    with open(os.path.join(out_dir, 'p7_per_record.csv'), 'w', newline='', encoding='utf-8') as fh:
        w = csv.writer(fh)
        w.writerow(['запись', 'выбрано', 'было', 'равноценны'] +
                   [f'{NAMES[c]}_{k}' for c in CHANNELS for k in ['балл', 'P', 'Q', 'R', 'S', 'T', 'сигнал/шум', 'исх_балл']])
        for r in rows:
            w.writerow([r['record'], NAMES[r['pick']], NAMES[r['orig_pick']], int(r['tie'])] +
                       [round(v, 4) for c in CHANNELS for v in
                        [r['scores'][c]] + [r['rates'][c][x] for x in 'PQRST'] + [r['snr'][c], r['orig_scores'][c]]])

    def _clean(o):
        if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.bool_,)): return bool(o)
        return o
    json.dump(_clean(summary), open(os.path.join(out_dir, 'p7_summary.json'), 'w', encoding='utf-8'),
              ensure_ascii=False, indent=1)
    print(f'\nРезультаты сохранены в папку {out_dir}')
    return summary


def _in_notebook():
    """True, если код выполняется в Jupyter / Google Colab (там нельзя разбирать
    аргументы командной строки: ядро передаёт свои служебные параметры)."""
    try:
        get_ipython  # noqa: F821  — определена только внутри Jupyter / Colab
        return True
    except NameError:
        return False


if __name__ == '__main__' and not _in_notebook():
    ap = argparse.ArgumentParser(description='П.7: выбор лучшего отведения')
    ap.add_argument('data_dir', help='папка с записями 00*_ecg.edf')
    ap.add_argument('--out', default='results_p7', help='папка для результатов')
    ap.add_argument('--no-synthetic', action='store_true', help='пропустить проверку на синтетике')
    a = ap.parse_args()
    run_p7(a.data_dir, a.out, with_synthetic=not a.no_synthetic)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 40.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 47.5 MB/s eta 0:00:00
  Created wheel for pyedflib: filename=pyedflib-0.1.42-cp313-cp313-linux_x86_64.whl size=2852049 sha256=60c17dc90842ae49259b8467e12425f890c8120fba29df480d8a9b09201ee895
  Stored in directory: /root/.cache/pip/wheels/ac/16/3e/5299c8a6b023bfd3d7204240ca5de940a49bede1edb30015ab
Successfully built pyedflib


In [2]:
run_p7('/content/data/')

Проверка детектора на синтетике...
  все зубцы               P: наход. 100% Q: наход. 100% S: наход. 100% T: наход. 100% Q_исх: наход. 0% S_исх: наход. 100%
  все зубцы, шум и дрейф  P: наход. 100% Q: наход. 99% S: наход. 100% T: наход. 100% Q_исх: наход. 0% S_исх: наход. 100%
  нет Q                   P: наход. 100% Q: ложн. 1% S: наход. 100% T: наход. 100% Q_исх: ложн. 100% S_исх: наход. 100%
  нет S                   P: наход. 100% Q: наход. 100% S: ложн. 2% T: наход. 100% Q_исх: наход. 0% S_исх: ложн. 100%
  нет Q и S               P: наход. 100% Q: ложн. 4% S: ложн. 2% T: наход. 100% Q_исх: ложн. 100% S_исх: ложн. 100%
  нет P                   P: ложн. 0% Q: наход. 100% S: наход. 100% T: наход. 100% Q_исх: наход. 0% S_исх: наход. 100%
  комплекс rS             P: наход. 100% Q: ложн. 0% S: наход. 100% T: наход. 100% Q_исх: ложн. 100% S_исх: наход. 100%
  сильный шум             P: наход. 98% Q: наход. 62% S: наход. 100% T: наход. 100% Q_исх: наход. 2% S_исх: наход. 100%
Проверка 

{'detector_check': [{'случай': 'все зубцы',
   'P': ('найдено', np.float64(1.0)),
   'Q': ('найдено', np.float64(1.0)),
   'S': ('найдено', np.float64(1.0)),
   'T': ('найдено', np.float64(1.0)),
   'Q_исх': ('найдено', np.float64(0.0)),
   'S_исх': ('найдено', np.float64(1.0))},
  {'случай': 'все зубцы, шум и дрейф',
   'P': ('найдено', np.float64(1.0)),
   'Q': ('найдено', np.float64(0.9875)),
   'S': ('найдено', np.float64(1.0)),
   'T': ('найдено', np.float64(1.0)),
   'Q_исх': ('найдено', np.float64(0.0)),
   'S_исх': ('найдено', np.float64(1.0))},
  {'случай': 'нет Q',
   'P': ('найдено', np.float64(1.0)),
   'Q': ('ложных', 0.0125),
   'S': ('найдено', np.float64(1.0)),
   'T': ('найдено', np.float64(1.0)),
   'Q_исх': ('ложных', 1.0),
   'S_исх': ('найдено', np.float64(1.0))},
  {'случай': 'нет S',
   'P': ('найдено', np.float64(1.0)),
   'Q': ('найдено', np.float64(1.0)),
   'S': ('ложных', 0.025),
   'T': ('найдено', np.float64(1.0)),
   'Q_исх': ('найдено', np.float64(0.0)),